# Week 1: Can we plan a tour through Europe?

A travelling salesperson wants to visit several European cities and return home while travelling as little as possible. This is the **Travelling Salesperson Problem (TSP)**.

We use it to unpack five recurring ideas in AI:

- **representation:** how a possible solution is stored,
- **objective function:** how a solution is scored,
- **search space:** all solutions an algorithm could inspect,
- **heuristic:** a useful rule that is not guaranteed to be optimal,
- **local search:** improving a solution through small changes.

> The goal is not to solve the old mandatory assignment. The goal is to build intuition and vocabulary for the course.

## 0. Before we start

Discuss with the person next to you:

1. What information does an algorithm need to solve this problem?
2. What should count as a *good* solution?
3. Would you expect one correct algorithm, or several competing approaches?

In [ ]:
from pathlib import Path
from math import factorial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(3050)

## 1. Meet the data

The file contains the pairwise road distances between 24 European cities. Rows and columns name cities; each cell is a distance in kilometres.

In [ ]:
candidate_paths = [
    Path('../datasets/european_cities.csv'),  # notebook opened inside week01/
    Path('datasets/european_cities.csv'),     # notebook run from repository root
    Path('european_cities.csv'),              # file placed beside the notebook
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find datasets/european_cities.csv')

distances = pd.read_csv(data_path, sep=';')
distances.index = distances.columns

print(f'{len(distances)} cities loaded from {data_path}')
distances.iloc[:6, :6].round(0)

### Sanity checks

Before using a dataset, ask whether it behaves as expected.

- Should the distance from a city to itself be zero?
- Should Oslo-to-Paris equal Paris-to-Oslo? (Oslo is not in this dataset, but the principle remains.)
- Which pair of cities do you predict is closest?

In [ ]:
D = distances.to_numpy()
print('All diagonal entries are zero:', np.allclose(np.diag(D), 0))
print('The matrix is symmetric:', np.allclose(D, D.T))

without_diagonal = D.copy()
np.fill_diagonal(without_diagonal, np.inf)
i, j = np.unravel_index(np.argmin(without_diagonal), D.shape)
print(f'Closest pair: {distances.index[i]} and {distances.columns[j]} '      f'({D[i, j]:.0f} km)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(D, cmap='viridis')
ax.set_xticks(range(len(distances)), distances.columns, rotation=90)
ax.set_yticks(range(len(distances)), distances.index)
ax.set_title('Distance between European cities (km)')
fig.colorbar(image, ax=ax, label='km')
plt.tight_layout()
plt.show()

## 2. Representation: what is a solution?

A tour can be represented as an ordered list of city names. The salesperson follows the list and finally returns to the first city.

Below, predict which edge is easy to forget when calculating the total length.

In [ ]:
example_tour = ['London', 'Paris', 'Brussels', 'Berlin', 'London']

def path_length(path, distance_table=distances):
    return sum(distance_table.loc[a, b] for a, b in zip(path[:-1], path[1:]))

for a, b in zip(example_tour[:-1], example_tour[1:]):
    print(f'{a:10s} -> {b:10s}: {distances.loc[a, b]:6.0f} km')
print(f'Total: {path_length(example_tour):.0f} km')

### From a path to a tour

We now write a reusable objective function. Smaller is better. This number plays the role often called **cost**, **loss**, or (after changing its sign) **fitness**.

In [ ]:
def tour_length(tour, distance_table=distances):
    closed_tour = list(tour) + [tour[0]]
    return path_length(closed_tour, distance_table)

cities = list(distances.columns[:10])
alphabetical_tour = cities.copy()
print('Tour:', ' -> '.join(alphabetical_tour + [alphabetical_tour[0]]))
print(f'Length: {tour_length(alphabetical_tour):,.0f} km')

## 3. Why not inspect every tour?

For $n$ cities there are roughly $(n-1)!/2$ distinct round trips when the starting city, direction, and rotations are treated as equivalent.

First vote: at one million tours per second, could we inspect every tour of all 24 cities during this session?

In [ ]:
def distinct_tours(n):
    return factorial(n - 1) // 2

rows = []
for n in [5, 10, 15, 20, 24]:
    tours = distinct_tours(n)
    years = tours / 1_000_000 / (60 * 60 * 24 * 365.25)
    rows.append({'cities': n, 'distinct tours': f'{tours:,}',
                 'years at 1M tours/s': f'{years:.2e}'})
pd.DataFrame(rows)

This explosive growth is **combinatorial explosion**. AI search often aims for a useful answer without inspecting every possible answer.

## 4. Baseline: random search

A baseline gives us something honest to beat. Random search repeatedly shuffles the cities and remembers the best tour seen so far.

Predict the shape of the best-so-far curve before running the cell. Can it ever become worse?

In [ ]:
def random_search(cities, iterations=2_000):
    best_tour = list(cities)
    best_length = tour_length(best_tour)
    history = []

    for _ in range(iterations):
        candidate = list(rng.permutation(cities))
        candidate_length = tour_length(candidate)
        if candidate_length < best_length:
            best_tour, best_length = candidate, candidate_length
        history.append(best_length)
    return best_tour, best_length, history

random_tour, random_length, random_history = random_search(cities)
print(f'Best random tour: {random_length:,.0f} km')

plt.figure(figsize=(8, 4))
plt.plot(random_history)
plt.xlabel('Tours evaluated')
plt.ylabel('Best distance so far (km)')
plt.title('Random search improves only when it gets lucky')
plt.show()

## 5. Heuristic: always visit the nearest unvisited city

A **greedy** algorithm makes the choice that looks best right now. It uses problem knowledge, but never reconsiders an earlier decision.

Discuss: why might a sequence of locally good choices produce a globally poor tour?

In [ ]:
def nearest_neighbour(cities, start):
    unvisited = set(cities) - {start}
    tour = [start]
    while unvisited:
        current = tour[-1]
        next_city = min(unvisited, key=lambda city: distances.loc[current, city])
        tour.append(next_city)
        unvisited.remove(next_city)
    return tour

greedy_tour = nearest_neighbour(cities, start=cities[0])
greedy_length = tour_length(greedy_tour)
print('Greedy tour:', ' -> '.join(greedy_tour + [greedy_tour[0]]))
print(f'Greedy length: {greedy_length:,.0f} km')
print(f'Best random length: {random_length:,.0f} km')

### Does the starting city matter?

A deterministic algorithm can still give different answers when its initial state changes.

In [ ]:
greedy_results = []
for start in cities:
    tour = nearest_neighbour(cities, start)
    greedy_results.append((start, tour_length(tour), tour))

greedy_results.sort(key=lambda result: result[1])
pd.DataFrame([(start, length) for start, length, _ in greedy_results],
             columns=['Starting city', 'Tour length (km)']).round(0)

## 6. Local search: improve a tour by swapping two cities

Local search starts with one complete solution, creates a nearby candidate, and keeps improvements. Here, a **neighbour** is produced by swapping two cities.

The algorithm can stop even when a better tour exists: it may be stuck in a **local optimum**.

In [ ]:
def swap_hill_climb(initial_tour):
    current = list(initial_tour)
    current_length = tour_length(current)
    history = [current_length]

    improved = True
    while improved:
        improved = False
        for i in range(len(current)):
            for j in range(i + 1, len(current)):
                neighbour = current.copy()
                neighbour[i], neighbour[j] = neighbour[j], neighbour[i]
                neighbour_length = tour_length(neighbour)
                if neighbour_length < current_length:
                    current, current_length = neighbour, neighbour_length
                    history.append(current_length)
                    improved = True
                    break
            if improved:
                break
    return current, current_length, history

best_greedy_tour = greedy_results[0][2]
improved_tour, improved_length, improvement_history = swap_hill_climb(best_greedy_tour)
print(f'Before local search: {tour_length(best_greedy_tour):,.0f} km')
print(f'After local search:  {improved_length:,.0f} km')
print('Improved tour:', ' -> '.join(improved_tour + [improved_tour[0]]))

In [ ]:
methods = ['Alphabetical', 'Random search', 'Greedy', 'Greedy + local search']
lengths = [tour_length(alphabetical_tour), random_length,
           tour_length(best_greedy_tour), improved_length]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(methods, lengths, color=['grey', '#4c78a8', '#f58518', '#54a24b'])
ax.bar_label(bars, labels=[f'{value:,.0f} km' for value in lengths], padding=3)
ax.set_ylabel('Tour length (lower is better)')
ax.set_title('Different search strategies, different solutions')
ax.set_ylim(0, max(lengths) * 1.18)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 7. Pair experiment

Choose **one** investigation and state your prediction before running code:

1. Increase from 10 to all 24 cities. Which method changes most?
2. Run hill climbing from five random starting tours. Does it always reach the same answer?
3. Change the random-search budget from 100 to 10,000 evaluations. Plot the result.
4. Make a different neighbourhood: reverse a section rather than swapping two cities.

Be ready to explain the result using at least two terms from today's vocabulary.

In [ ]:
# Your experiment here


## 8. Exit ticket

Explain in one sentence each:

1. What is the **representation** in this problem?
2. What is the **objective function**?
3. Why is exhaustive search impractical for 24 cities?
4. What is the difference between a heuristic and a guarantee?
5. Why can hill climbing stop before finding the globally shortest tour?

### Bridge to the course

Later methods—including evolutionary algorithms—change how candidates are generated and selected. The same basic questions remain: **How do we represent a solution? How do we score it? How do we explore a huge search space?**

## Optional IN4050 challenge

The same physical round trip can be represented several ways. For example, rotations and reversed tours are equivalent. Derive why fixing one starting city and treating reversal as equivalent reduces $n!$ sequences to $(n-1)!/2$ distinct tours.